In [3]:
import os
from getpass import getpass

from dotenv import load_dotenv
from crewai import Agent, Task, Crew, LLM
import yfinance as yf
from duckduckgo_search import DDGS
from crewai.tools import BaseTool

# Load environment variables from the local .env file if present.
load_dotenv()

# Configure the Gemini API key. Prompt the user if it is not set in the environment.
api_key = os.getenv("GEMINI_API_KEY")
if not api_key:
    api_key = getpass("Enter your Gemini API key: ")
os.environ["GEMINI_API_KEY"] = api_key

# Initialize the CrewAI LLM client for Gemini.
llm = LLM(
    model="gemini/gemini-2.5-flash",
    api_key=api_key,
    temperature=0.7,
)


class StockSearchTool(BaseTool):
    name: str = "StockNewsSearcher"
    description: str = "Search for the latest news and updates about a stock using DuckDuckGo"

    def _run(self, query: str) -> str:
        # Use DuckDuckGo to gather a compact set of recent news results for the requested stock.
        try:
            with DDGS() as ddgs:
                results = ddgs.text(query, max_results=2)
            if not results:
                return "No news results were found."
            return "\n".join([r.get("body", "") for r in results if r.get("body")])
        except Exception as exc:
            return f"News search failed: {exc}"


class YahooFinanceTool(BaseTool):
    name: str = "YahooFinanceFetcher"
    description: str = "Get the latest 1-month stock price history for a given ticker using yFinance"

    def _run(self, ticker: str) -> str:
        # Fetch the most recent price history to support the price analysis task.
        try:
            stock = yf.Ticker(ticker)
            hist = stock.history(period="1mo")
            if hist.empty:
                return f"No price data available for {ticker}."
            return hist.tail(3).to_string()
        except Exception as exc:
            return f"Price data lookup failed: {exc}"


# Step 6: Define the two specialized agents for the multi-agent workflow.
stock_analyst = Agent(
    role="Stock Analyst",
    goal="Analyze recent stock data and news",
    backstory="Expert in financial trends, macro indicators, and company performance",
    verbose=True,
    allow_delegation=False,
    llm=llm,
)

report_writer = Agent(
    role="Report Generator",
    goal="Write investor-friendly summaries of stock analysis",
    backstory="Professional writer with expertise in finance reporting",
    verbose=True,
    allow_delegation=False,
    llm=llm,
)

# Step 7: Create tasks for news gathering, trend analysis, and report generation.
search_tool = StockSearchTool()
finance_tool = YahooFinanceTool()

search_task = Task(
    description="Search latest news and updates about the stock 'AAPL' using DuckDuckGo.",
    expected_output="Summarized news highlights for Apple stock.",
    agent=stock_analyst,
    tools=[search_tool],
)

analysis_task = Task(
    description="Analyze Apple stock price trends using yFinance.",
    expected_output="Key trends and technical highlights for the past month.",
    agent=stock_analyst,
    tools=[finance_tool],
)

report_task = Task(
    description="Write a clean investor report using previous analysis and news insights.",
    expected_output="Concise report with market summary and investment outlook.",
    agent=report_writer,
)

# Step 8: Assemble the crew so the agents can work together in sequence.
crew = Crew(
    agents=[stock_analyst, report_writer],
    tasks=[search_task, analysis_task, report_task],
    verbose=False,
)


In [4]:
# Step 9: Run the agent crew and print the final investor report.
result = await crew.kickoff_async()

print("\n📊 Final Stock Analysis Report:\n")
print(result)


╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Stock Analyst                                                                                           │
│                                                                                                                 │
│  Task: Search latest news and updates about the stock 'AAPL' using DuckDuckGo.                                  │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

C:\Users\Shubham\AppData\Local\Temp\ipykernel_25060\3192548554.py:34: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:


Tool stock_news_searcher executed with result: No news results were found....
[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Stock Analyst                                                                                           │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  No news results were found for AAPL. Therefore, I cannot provide summarized news highlights for Apple stock.   │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

c:\Shubham\ProjectsAgenticAI\StockAnalysisCrewAI\saa_venv\Lib\site-packages\crewai\crew.py:1633: DeprecationWarning: deprecated
  if hasattr(agent, "allow_code_execution") and getattr(
c:\Shubham\ProjectsAgenticAI\StockAnalysisCrewAI\saa_venv\Lib\site-packages\crewai\crew.py:1633: DeprecationWarning: deprecated
  if hasattr(agent, "allow_code_execution") and getattr(
c:\Shubham\ProjectsAgenticAI\StockAnalysisCrewAI\saa_venv\Lib\site-packages\crewai\crew.py:1640: DeprecationWarning: deprecated
  and hasattr(agent, "multimodal")
c:\Shubham\ProjectsAgenticAI\StockAnalysisCrewAI\saa_venv\Lib\site-packages\crewai\crew.py:1641: DeprecationWarning: deprecated
  and getattr(agent, "multimodal", False)


...

c:\Shubham\ProjectsAgenticAI\StockAnalysisCrewAI\saa_venv\Lib\site-packages\crewai\crew.py:1633: DeprecationWarning: deprecated
  if hasattr(agent, "allow_code_execution") and getattr(
c:\Shubham\ProjectsAgenticAI\StockAnalysisCrewAI\saa_venv\Lib\site-packages\crewai\crew.py:1633: DeprecationWarning: deprecated
  if hasattr(agent, "allow_code_execution") and getattr(
c:\Shubham\ProjectsAgenticAI\StockAnalysisCrewAI\saa_venv\Lib\site-packages\crewai\crew.py:1640: DeprecationWarning: deprecated
  and hasattr(agent, "multimodal")
c:\Shubham\ProjectsAgenticAI\StockAnalysisCrewAI\saa_venv\Lib\site-packages\crewai\crew.py:1641: DeprecationWarning: deprecated
  and getattr(agent, "multimodal", False)


╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Report Generator                                                                                        │
│                                                                                                                 │
│  Task: Write a clean investor report using previous analysis and news insights.                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Report Generator                                                                                        │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **Apple (AAPL) Investor Report**                                                                               │
│                                                                                                                 │
│  **Report Date:** October 26, 2023                                                                              │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  **Market Summary (Based on Limited Data)**                                                                     │
│                                                                                                                 │
│  Analysis of Apple (AAPL) stock is currently constrained by the available data. The provided information        │
│  covers only three future-dated trading days (June 30, 2026, July 1, 2026, and July 2, 2026), rather than       │
│  recent historical performance. Furthermore, no current news insights for AAPL were found to supplement this    │
│  report.                                                                                                        │
│                                                                                                                 │
│  Based strictly on the provided three-day snapshot:                                                             │
│                                                                                                                 │
│  *   **Upward Price Momentum:** Over this brief period, AAPL demonstrated a clear positive trend. The stock     │
│  consistently opened higher each day, reaching higher highs and closing significantly above its opening price   │
│  on July 2, 2026 ($308.63 close vs. $289.36 close on June 30, 2026).                                            │
│  *   **Volume Fluctuation:** Trading volume fluctuated, with the highest volume aligning with the largest       │
│  single-day price increase on July 2, 2026.                                                                     │
│                                                                                                                 │
│  Due to the extremely limited scope (three future-dated days) and absence of current news, it is not possible   │
│  to conduct a comprehensive technical analysis, identify key support/resistance levels, calculate moving        │
│  averages, or assess broader market sentiment and historical trends for Apple stock at this time.               │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  **Investment Outlook**                                                                                         │
│                                                                                                                 │
│  Given the severe limitations of the provided data—bein


📊 Final Stock Analysis Report:

**Apple (AAPL) Investor Report**

**Report Date:** October 26, 2023

---

**Market Summary (Based on Limited Data)**

Analysis of Apple (AAPL) stock is currently constrained by the available data. The provided information covers only three future-dated trading days (June 30, 2026, July 1, 2026, and July 2, 2026), rather than recent historical performance. Furthermore, no current news insights for AAPL were found to supplement this report.

Based strictly on the provided three-day snapshot:

*   **Upward Price Momentum:** Over this brief period, AAPL demonstrated a clear positive trend. The stock consistently opened higher each day, reaching higher highs and closing significantly above its opening price on July 2, 2026 ($308.63 close vs. $289.36 close on June 30, 2026).
*   **Volume Fluctuation:** Trading volume fluctuated, with the highest volume aligning with the largest single-day price increase on July 2, 2026.

Due to the extremely limited scope (th